In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

# Load all data
races = pd.read_csv("../data/raw/races.csv")
lap_times = pd.read_csv("../data/raw/lap_times.csv")
results = pd.read_csv("../data/raw/results.csv")
circuits = pd.read_csv("../data/raw/circuits.csv")
drivers = pd.read_csv("../data/raw/drivers.csv")
constructors = pd.read_csv("../data/raw/constructors.csv")
pit_stops = pd.read_csv("../data/raw/pit_stops.csv")

print("✓ All data loaded")

✓ All data loaded


In [2]:
print("\n" + "="*60)
print("CALCULATING CIRCUIT STATISTICS")
print("="*60)

circuit_stats = {}

for circuit_id in circuits['circuitId'].unique():
    circuit_name = circuits[circuits['circuitId'] == circuit_id]['name'].values[0]
    
    # Get races at this circuit
    circuit_races = races[races['circuitId'] == circuit_id]
    race_ids = circuit_races['raceId'].values
    
    # Avg lap time
    circuit_laps = lap_times[lap_times['raceId'].isin(race_ids)]
    circuit_laps = circuit_laps[circuit_laps['milliseconds'].notna()]
    avg_lap_ms = circuit_laps['milliseconds'].mean()
    avg_lap_sec = avg_lap_ms / 1000 if not np.isnan(avg_lap_ms) else None
    
    # Top-10 rate & podium rate
    circuit_results = results[results['raceId'].isin(race_ids)]
    circuit_results['position_numeric'] = pd.to_numeric(circuit_results['positionText'], errors='coerce')
    circuit_results = circuit_results[circuit_results['position_numeric'].notna()]
    
    if len(circuit_results) > 0:
        top10_rate = ((circuit_results['position_numeric'] <= 10).sum() / len(circuit_results)) * 100
        podium_rate = ((circuit_results['position_numeric'] <= 3).sum() / len(circuit_results)) * 100
    else:
        top10_rate = 0
        podium_rate = 0
    
    circuit_stats[circuit_name] = {
        'avg_lap_time_sec': round(avg_lap_sec, 2) if avg_lap_sec else None,
        'top10_rate': round(top10_rate, 1),
        'podium_rate': round(podium_rate, 1),
        'races_count': len(circuit_races)
    }

print(f"✓ Calculated stats for {len(circuit_stats)} circuits")
print("\nSample circuits:")
for i, (name, stats) in enumerate(list(circuit_stats.items())[:5]):
    print(f"  {name}: {stats['avg_lap_time_sec']}s avg, {stats['top10_rate']}% top-10")

# Save
Path("../data/processed").mkdir(exist_ok=True)
with open("../data/processed/circuit_stats.json", "w") as f:
    json.dump(circuit_stats, f, indent=2)
print("\n✓ Circuit stats saved")


CALCULATING CIRCUIT STATISTICS


C:\Users\ACER\AppData\Local\Temp\ipykernel_19372\3826831189.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  circuit_results['position_numeric'] = pd.to_numeric(circuit_results['positionText'], errors='coerce')
C:\Users\ACER\AppData\Local\Temp\ipykernel_19372\3826831189.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  circuit_results['position_numeric'] = pd.to_numeric(circuit_results['positionText'], errors='coerce')
C:\Users\ACER\AppData\Local\Temp\ipykernel_19372\3826831189.py:22: SettingWithCopy

✓ Calculated stats for 77 circuits

Sample circuits:
  Albert Park Grand Prix Circuit: 98.99s avg, 72.6% top-10
  Sepang International Circuit: 110.05s avg, 63.8% top-10
  Bahrain International Circuit: 99.78s avg, 56.9% top-10
  Circuit de Barcelona-Catalunya: 88.47s avg, 62.3% top-10
  Istanbul Park: 94.79s avg, 54.5% top-10

✓ Circuit stats saved


C:\Users\ACER\AppData\Local\Temp\ipykernel_19372\3826831189.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  circuit_results['position_numeric'] = pd.to_numeric(circuit_results['positionText'], errors='coerce')
C:\Users\ACER\AppData\Local\Temp\ipykernel_19372\3826831189.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  circuit_results['position_numeric'] = pd.to_numeric(circuit_results['positionText'], errors='coerce')


In [3]:
print("\n" + "="*60)
print("CALCULATING DRIVER STATISTICS (Global + Circuit-Specific)")
print("="*60)

# Global driver stats (keep this)
driver_stats = {}

# Circuit-specific driver stats (NEW)
driver_stats_by_circuit = {}

for driver_id in drivers['driverId'].unique():
    forename = drivers[drivers['driverId'] == driver_id]['forename'].values[0]
    surname = drivers[drivers['driverId'] == driver_id]['surname'].values[0]
    driver_name = f"{forename} {surname}"  # Full name
    
    driver_results = results[results['driverId'] == driver_id]
    driver_results['position_numeric'] = pd.to_numeric(driver_results['positionText'], errors='coerce')
    driver_results = driver_results[driver_results['position_numeric'].notna()]
    
    if len(driver_results) > 0:
        avg_finish = driver_results['position_numeric'].mean()
        top10_rate = ((driver_results['position_numeric'] <= 10).sum() / len(driver_results)) * 100
        podium_rate = ((driver_results['position_numeric'] <= 3).sum() / len(driver_results)) * 100
        wins = ((driver_results['position_numeric'] == 1).sum())
    else:
        avg_finish = None
        top10_rate = 0
        podium_rate = 0
        wins = 0
    
    # Global stats
    driver_stats[driver_name] = {
        'avg_finish': round(avg_finish, 2) if avg_finish else None,
        'top10_rate': round(top10_rate, 1),
        'podium_rate': round(podium_rate, 1),
        'wins': int(wins),
        'races': len(driver_results)
    }
    
    # Circuit-specific stats (NEW) ↓
    driver_stats_by_circuit[driver_name] = {}
    
    for circuit_id in circuits['circuitId'].unique():
        circuit_name = circuits[circuits['circuitId'] == circuit_id]['name'].values[0]
        circuit_races = races[races['circuitId'] == circuit_id]
        race_ids = circuit_races['raceId'].values
        
        circuit_driver_results = driver_results[driver_results['raceId'].isin(race_ids)]
        
        if len(circuit_driver_results) > 0:
            circuit_avg_finish = circuit_driver_results['position_numeric'].mean()
            circuit_top10_rate = ((circuit_driver_results['position_numeric'] <= 10).sum() / len(circuit_driver_results)) * 100
            circuit_wins = ((circuit_driver_results['position_numeric'] == 1).sum())
        else:
            circuit_avg_finish = None
            circuit_top10_rate = 0
            circuit_wins = 0
        
        driver_stats_by_circuit[driver_name][circuit_name] = {
            'avg_finish': round(circuit_avg_finish, 2) if circuit_avg_finish else None,
            'top10_rate': round(circuit_top10_rate, 1),
            'wins': int(circuit_wins),
            'races': len(circuit_driver_results)
        }

print(f"✓ Calculated stats for {len(driver_stats)} drivers")
print("✓ Calculated circuit-specific stats for drivers")

# Save global (keep this)
with open("../data/processed/driver_stats.json", "w") as f:
    json.dump(driver_stats, f, indent=2)

# Save circuit-specific (NEW)
with open("../data/processed/driver_stats_by_circuit.json", "w") as f:
    json.dump(driver_stats_by_circuit, f, indent=2)
    
print("✓ Driver stats saved (global + by circuit)")


CALCULATING DRIVER STATISTICS (Global + Circuit-Specific)


C:\Users\ACER\AppData\Local\Temp\ipykernel_19372\877448569.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  driver_results['position_numeric'] = pd.to_numeric(driver_results['positionText'], errors='coerce')
C:\Users\ACER\AppData\Local\Temp\ipykernel_19372\877448569.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  driver_results['position_numeric'] = pd.to_numeric(driver_results['positionText'], errors='coerce')
C:\Users\ACER\AppData\Local\Temp\ipykernel_19372\877448569.py:17: SettingWithCopyWarning

✓ Calculated stats for 859 drivers
✓ Calculated circuit-specific stats for drivers
✓ Driver stats saved (global + by circuit)


In [4]:
print("\n" + "="*60)
print("CALCULATING TEAM/CONSTRUCTOR STATISTICS (Global + Circuit-Specific)")
print("="*60)

# Global team stats (keep this)
team_stats = {}

# Circuit-specific team stats (NEW)
team_stats_by_circuit = {}

for constructor_id in constructors['constructorId'].unique():
    constructor_name = constructors[constructors['constructorId'] == constructor_id]['name'].values[0]
    
    team_results = results[results['constructorId'] == constructor_id]
    team_results['position_numeric'] = pd.to_numeric(team_results['positionText'], errors='coerce')
    team_results = team_results[team_results['position_numeric'].notna()]
    
    if len(team_results) > 0:
        avg_finish = team_results['position_numeric'].mean()
        top10_rate = ((team_results['position_numeric'] <= 10).sum() / len(team_results)) * 100
        podium_rate = ((team_results['position_numeric'] <= 3).sum() / len(team_results)) * 100
        wins = ((team_results['position_numeric'] == 1).sum())
    else:
        avg_finish = None
        top10_rate = 0
        podium_rate = 0
        wins = 0
    
    # Global stats
    team_stats[constructor_name] = {
        'avg_finish': round(avg_finish, 2) if avg_finish else None,
        'top10_rate': round(top10_rate, 1),
        'podium_rate': round(podium_rate, 1),
        'wins': int(wins),
        'races': len(team_results)
    }
    
    # Circuit-specific stats (NEW) ↓
    team_stats_by_circuit[constructor_name] = {}
    
    for circuit_id in circuits['circuitId'].unique():
        circuit_name = circuits[circuits['circuitId'] == circuit_id]['name'].values[0]
        circuit_races = races[races['circuitId'] == circuit_id]
        race_ids = circuit_races['raceId'].values
        
        circuit_team_results = team_results[team_results['raceId'].isin(race_ids)]
        
        if len(circuit_team_results) > 0:
            circuit_avg_finish = circuit_team_results['position_numeric'].mean()
            circuit_top10_rate = ((circuit_team_results['position_numeric'] <= 10).sum() / len(circuit_team_results)) * 100
            circuit_wins = ((circuit_team_results['position_numeric'] == 1).sum())
        else:
            circuit_avg_finish = None
            circuit_top10_rate = 0
            circuit_wins = 0
        
        team_stats_by_circuit[constructor_name][circuit_name] = {
            'avg_finish': round(circuit_avg_finish, 2) if circuit_avg_finish else None,
            'top10_rate': round(circuit_top10_rate, 1),
            'wins': int(circuit_wins),
            'races': len(circuit_team_results)
        }

print(f"✓ Calculated stats for {len(team_stats)} teams")
print("✓ Calculated circuit-specific stats for teams")

# Save global (keep this)
with open("../data/processed/team_stats.json", "w") as f:
    json.dump(team_stats, f, indent=2)

# Save circuit-specific (NEW)
with open("../data/processed/team_stats_by_circuit.json", "w") as f:
    json.dump(team_stats_by_circuit, f, indent=2)
    
print("✓ Team stats saved (global + by circuit)")


CALCULATING TEAM/CONSTRUCTOR STATISTICS (Global + Circuit-Specific)


C:\Users\ACER\AppData\Local\Temp\ipykernel_19372\1895817344.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  team_results['position_numeric'] = pd.to_numeric(team_results['positionText'], errors='coerce')
C:\Users\ACER\AppData\Local\Temp\ipykernel_19372\1895817344.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  team_results['position_numeric'] = pd.to_numeric(team_results['positionText'], errors='coerce')
C:\Users\ACER\AppData\Local\Temp\ipykernel_19372\1895817344.py:15: SettingWithCopyWarning: 
A 

✓ Calculated stats for 212 teams
✓ Calculated circuit-specific stats for teams
✓ Team stats saved (global + by circuit)


In [5]:
print("\n" + "="*60)
print("PIT STOP ANALYSIS")
print("="*60)

# This shows how pit stops could be analyzed
pit_stats = {}

for race_id in results['raceId'].unique():
    race_pit_stops = pit_stops[pit_stops['raceId'] == race_id]
    race_results = results[results['raceId'] == race_id]
    
    for driver_id in race_results['driverId'].unique():
        driver_pit_stops = race_pit_stops[race_pit_stops['driverId'] == driver_id]
        pit_count = len(driver_pit_stops)
        
        driver_result = race_results[race_results['driverId'] == driver_id]
        position = pd.to_numeric(driver_result['positionText'].values[0] if len(driver_result) > 0 else np.nan, errors='coerce')
        
        if not np.isnan(position):
            key = int(pit_count)
            if key not in pit_stats:
                pit_stats[key] = {'count': 0, 'top10': 0, 'avg_position': []}
            pit_stats[key]['count'] += 1
            if position <= 10:
                pit_stats[key]['top10'] += 1
            pit_stats[key]['avg_position'].append(position)

print("\nTop-10 finish rate by pit stop count:")
for pit_count in sorted(pit_stats.keys()):
    stats = pit_stats[pit_count]
    top10_pct = (stats['top10'] / stats['count'] * 100) if stats['count'] > 0 else 0
    avg_pos = np.mean(stats['avg_position'])
    print(f"  {pit_count} stops: {top10_pct:.1f}% top-10, avg finish P{avg_pos:.1f}")

print("\n✓ Pit stop analysis complete")


PIT STOP ANALYSIS

Top-10 finish rate by pit stop count:
  0 stops: 75.4% top-10, avg finish P7.4
  1 stops: 64.5% top-10, avg finish P8.6
  2 stops: 55.8% top-10, avg finish P9.5
  3 stops: 49.7% top-10, avg finish P10.4
  4 stops: 48.0% top-10, avg finish P10.4
  5 stops: 54.7% top-10, avg finish P9.8
  6 stops: 56.5% top-10, avg finish P9.9
  7 stops: 0.0% top-10, avg finish P13.7

✓ Pit stop analysis complete


In [6]:
print("\n" + "="*60)
print("STATISTICS GENERATION COMPLETE")
print("="*60)
print("\nFiles saved:")
print("  ✓ circuit_stats.json")
print("  ✓ driver_stats.json")
print("  ✓ team_stats.json")
print("  ✓ driver_stats_by_circuit.json")
print("  ✓ team_stats_by_circuit.json")
print("\nReady to use in app.py!")


STATISTICS GENERATION COMPLETE

Files saved:
  ✓ circuit_stats.json
  ✓ driver_stats.json
  ✓ team_stats.json
  ✓ driver_stats_by_circuit.json
  ✓ team_stats_by_circuit.json

Ready to use in app.py!
